## JOINS

Bring column/data from 2 dataframes together based on a matching key.

## AGGREGATION

Takerows and summarize them into fewer rows.

## creating dataframe's

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark=SparkSession.builder.appName("JOINS Practice").master("local[*]").getOrCreate()

In [ ]:
customers=spark.createDataFrame(
    [
        (1,"Alice"),
        (2,"Bob"),
        (3,"Charile"),
        (4, "David")
    ] , ["customer_id" , "name"]
)
customers.show()

+-----------+-------+
|customer_id|   name|
+-----------+-------+
|          1|  Alice|
|          2|    Bob|
|          3|Charile|
|          4|  David|
+-----------+-------+



In [ ]:
orders=spark.createDataFrame(
    [
        (101, 1, 500),
        (102 , 1, 200),
        (103 , 2 , 300),
        (104 , 5 , 100)
    ] , ["order_id" , "customer_id" , "amount"]
)
orders.show()

+--------+-----------+------+
|order_id|customer_id|amount|
+--------+-----------+------+
|     101|          1|   500|
|     102|          1|   200|
|     103|          2|   300|
|     104|          5|   100|
+--------+-----------+------+



In [ ]:
#another type of creating the dataframe
from pyspark.sql import Row
employees=spark.createDataFrame([
    Row(id=101 , name="ram"),
    Row(id=102 , name="sita")
])

employees.show()

+---+----+
| id|name|
+---+----+
|101| ram|
|102|sita|
+---+----+



## Inner join

In [ ]:
cust_order=customers.join(orders , on="customer_id" , how="inner")
cust_order.show()

+-----------+-----+--------+------+
|customer_id| name|order_id|amount|
+-----------+-----+--------+------+
|          1|Alice|     101|   500|
|          1|Alice|     102|   200|
|          2|  Bob|     103|   300|
+-----------+-----+--------+------+



## Left join

In [ ]:
cust_order=customers.join(orders , on="customer_id" , how="left")
cust_order.show()

+-----------+-------+--------+------+
|customer_id|   name|order_id|amount|
+-----------+-------+--------+------+
|          1|  Alice|     102|   200|
|          1|  Alice|     101|   500|
|          2|    Bob|     103|   300|
|          3|Charile|    NULL|  NULL|
|          4|  David|    NULL|  NULL|
+-----------+-------+--------+------+



## Right Join

In [ ]:
cust_order=customers.join(orders , on="customer_id" , how="right")
cust_order.show()

+-----------+-----+--------+------+
|customer_id| name|order_id|amount|
+-----------+-----+--------+------+
|          1|Alice|     101|   500|
|          1|Alice|     102|   200|
|          5| NULL|     104|   100|
|          2|  Bob|     103|   300|
+-----------+-----+--------+------+



## What if the column name's are different in 2 table's

but this is creating an duplicate solumn with customer_id again

when you are explicitly joining them like  customers.customer_id==orders.customer_id

In [ ]:
cust_ord=customers.join(orders , customers.customer_id==orders.customer_id ,"inner" )
cust_ord.show()

+-----------+-----+--------+-----------+------+
|customer_id| name|order_id|customer_id|amount|
+-----------+-----+--------+-----------+------+
|          1|Alice|     101|          1|   500|
|          1|Alice|     102|          1|   200|
|          2|  Bob|     103|          2|   300|
+-----------+-----+--------+-----------+------+



## sort the value's ( order by)

In [ ]:
import pyspark.sql.functions as sf

In [ ]:
cust_order=customers.join(orders , on="customer_id" , how="inner" ).sort(sf.desc(orders["amount"]))
cust_order.show()

+-----------+-----+--------+------+
|customer_id| name|order_id|amount|
+-----------+-----+--------+------+
|          1|Alice|     101|   500|
|          2|  Bob|     103|   300|
|          1|Alice|     102|   200|
+-----------+-----+--------+------+



## GROUP BY
group by groups the rows accoring to a column and then we cna perform the aggregate function in a easier way .

In [ ]:
cust_order1=cust_order.groupBy("name").sum("amount")
cust_order1.show()

+-----+-----------+
| name|sum(amount)|
+-----+-----------+
|  Bob|        300|
|Alice|        700|
+-----+-----------+



In [ ]:
cust_order2=cust_order.groupBy("customer_id").agg(
    sf.sum("amount").alias("total_amount"),
    sf.avg("amount").alias("average_amount"),
    sf.count("*").alias("order_count")
)
#or you can even do an
#from pyspark.sql.functions import sum , avg , count
# and without using sf we can write sum() avg() count()

cust_order2.show()

+-----------+------------+--------------+-----------+
|customer_id|total_amount|average_amount|order_count|
+-----------+------------+--------------+-----------+
|          1|         700|         350.0|          2|
|          2|         300|         300.0|          1|
+-----------+------------+--------------+-----------+



NULL AFTER THE LEFT JOIN

In [ ]:
cus=spark.createDataFrame(
    [
        (1, "Alice"),
        (2,"Bob"),
        (3,"Charlie")
    ],["customer_id", "name"]
)

In [ ]:
ors=spark.createDataFrame([
    (101,1,500),
    (102,1,200),
    (103,2,300)
],["order_id", "customer_id" , "amount"])

In [ ]:
cur_ors=cus.join(ors,on="customer_id", how="left")
cur_ors.show()

+-----------+-------+--------+------+
|customer_id|   name|order_id|amount|
+-----------+-------+--------+------+
|          1|  Alice|     102|   200|
|          1|  Alice|     101|   500|
|          3|Charlie|    NULL|  NULL|
|          2|    Bob|     103|   300|
+-----------+-------+--------+------+



In [ ]:
cur_ors_count_amount=cur_ors.groupBy("customer_id").agg(
    sf.count("amount").alias("order_count")
)
cur_ors_count_amount.show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|          1|          2|
|          3|          0|
|          2|          1|
+-----------+-----------+



In [ ]:
cur_ors.groupBy("customer_id").agg(sf.sum("amount").alias("total_amount")).show()

+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|          1|         700|
|          3|        NULL|
|          2|         300|
+-----------+------------+



In [ ]:
cur_ors.groupBy("customer_id").agg(sf.coalesce(sf.sum("amount"),sf.lit(0)).alias("total_amount")).show()

+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|          1|         700|
|          3|           0|
|          2|         300|
+-----------+------------+



You can do joins with multiple columns

In [ ]:
cus=spark.createDataFrame(
    [
        (1, "Alice","it"),
        (2,"Bob","hr"),
        (3,"Charlie","it")
    ],["customer_id", "name","department"]
)

In [ ]:
ors=spark.createDataFrame([
    (101,1,"Alice",500),
    (102,1,"Alice",200),
    (103,2,"Bob",300)
],["order_id", "customer_id" , "name" , "amount"])

In [ ]:
cus.join(ors, on=["customer_id", "name"], how="inner").show()

+-----------+-----+----------+--------+------+
|customer_id| name|department|order_id|amount|
+-----------+-----+----------+--------+------+
|          1|Alice|        it|     101|   500|
|          1|Alice|        it|     102|   200|
|          2|  Bob|        hr|     103|   300|
+-----------+-----+----------+--------+------+



In [ ]:
cus.join(ors, (cus.customer_id==ors.customer_id) & (cus.name==ors.name) , "left").show()

+-----------+-------+----------+--------+-----------+-----+------+
|customer_id|   name|department|order_id|customer_id| name|amount|
+-----------+-------+----------+--------+-----------+-----+------+
|          1|  Alice|        it|     102|          1|Alice|   200|
|          1|  Alice|        it|     101|          1|Alice|   500|
|          3|Charlie|        it|    NULL|       NULL| NULL|  NULL|
|          2|    Bob|        hr|     103|          2|  Bob|   300|
+-----------+-------+----------+--------+-----------+-----+------+



Alias make join more simple

In [ ]:
# c=cus.alias("cu") #crct
# o=ors.alias("or") #crct

c=cus
o=ors

# r=cu.join(or , "customer_id" , "left").show()  #wrong
r=c.join(o, on=["customer_id", "name"], how="inner")
r.show()

+-----------+-----+----------+--------+------+
|customer_id| name|department|order_id|amount|
+-----------+-----+----------+--------+------+
|          1|Alice|        it|     101|   500|
|          1|Alice|        it|     102|   200|
|          2|  Bob|        hr|     103|   300|
+-----------+-----+----------+--------+------+



GLOABAL AGGREAGATION

In [ ]:
ors.groupby().agg(sf.sum("amount").alias("total_amount")).show()
ors.agg(sf.sum("amount").alias("total_amount")).show()

+------------+
|total_amount|
+------------+
|        1000|
+------------+

+------------+
|total_amount|
+------------+
|        1000|
+------------+

